# AEMO offline RL workflow notebook

This notebook replaces the script-first AEMO workflow with an inspectable notebook flow for:

1. fetching + caching AEMO market data across multiple regions and time windows
2. sweeping multiple battery sizes
3. collecting rule / dispatch-replay / SB3 trajectories
4. exporting parquet logs and a DT-ready dataset
5. optionally launching Decision Transformer training

In [ ]:
# Repository bootstrap for relocated notebooks
from pathlib import Path
import os
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
for _extra_path in (REPO_ROOT, REPO_ROOT / "src"):
    _extra_str = str(_extra_path)
    if _extra_str not in sys.path:
        sys.path.insert(0, _extra_str)

DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
EVAL_OUTPUT_DIR = REPO_ROOT / "eval_output"
CONFIGS_DIR = REPO_ROOT / "configs"


In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

import polars as pl

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

In [2]:
from aemo_notebook_utils import (
    build_dispatch_selection,
    build_dt_dataset_from_logs,
    build_model_config,
    fetch_and_preprocess_aemo_scenarios,
    fit_aemo_global_stats,
    launch_dt_training,
    prepare_run_paths,
    resolve_battery_variants,
    resolve_dispatch_battery_life_cost,
    resolve_dispatch_replay_runs,
    run_rule_episodes,
    run_sb3_episodes,
    should_run_dispatch_for_scenario,
    validate_aemo_dt_dimensions,
    write_combined_episode_logs,
    write_json,
)

from dispatch_utils import run_dispatch_replay, list_known_batteries

/usr/local/lib/python3.10/dist-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
# Show all registered batteries with their historical DUID mappings.
# The "Key" column is a short name you can pass to show_dispatch_candidates(station_name=...).
battery_station_list = list_known_batteries()

In [4]:
REGION_ORDER = ['SA1', 'VIC1', 'QLD1', 'NSW1', 'TAS1']
REGISTRY_VIEW_COLUMNS = ['Key', 'StationName', 'Region', 'DUID', 'DispatchType', 'ValidFrom', 'ValidUntil']

for region in REGION_ORDER:
    region_rows = battery_station_list.filter(pl.col('Region') == region)
    if region_rows.height == 0:
        continue
    print(f'\n{region} stations:')
    print(region_rows.select([col for col in REGISTRY_VIEW_COLUMNS if col in region_rows.columns]))


SA1 stations:
shape: (11, 7)
┌───────────────┬───────────────┬────────┬──────────┬───────────────┬───────────────┬──────────────┐
│ Key           ┆ StationName   ┆ Region ┆ DUID     ┆ DispatchType  ┆ ValidFrom     ┆ ValidUntil   │
│ ---           ┆ ---           ┆ ---    ┆ ---      ┆ ---           ┆ ---           ┆ ---          │
│ str           ┆ str           ┆ str    ┆ str      ┆ str           ┆ datetime[μs]  ┆ datetime[μs] │
╞═══════════════╪═══════════════╪════════╪══════════╪═══════════════╪═══════════════╪══════════════╡
│ blyth         ┆ Blyth Battery ┆ SA1    ┆ BLYTHB1  ┆ bidirectional ┆ 2023-01-01    ┆ null         │
│               ┆ Energy        ┆        ┆          ┆               ┆ 00:00:00      ┆              │
│               ┆ Storage S…    ┆        ┆          ┆               ┆               ┆              │
│ bungama       ┆ Bungama       ┆ SA1    ┆ BUNGAMB1 ┆ bidirectional ┆ 2023-06-01    ┆ null         │
│               ┆ Battery       ┆        ┆          ┆        

## 1. Experiment configuration

Edit this cell first. The notebook is designed so you can inspect each intermediate object before moving to the next stage.

In [5]:
SCENARIOS = [
    {
        'label': 'nsw1_2021_2023',
        'region': 'NSW1',
        'start_date': datetime.fromisoformat('2021-01-01'),
        'end_date': datetime.fromisoformat('2023-04-01'),
    },
    {
        'label': 'qld1_2021_2023',
        'region': 'QLD1',
        'start_date': datetime.fromisoformat('2021-01-01'),
        'end_date': datetime.fromisoformat('2023-04-01'),
    },
    {
        'label': 'sa1_2022_2023',
        'region': 'SA1',
        'start_date': datetime.fromisoformat('2022-04-01'),
        'end_date': datetime.fromisoformat('2023-12-01'),
    },
    {
        'label': 'tas1_2021_2023',
        'region': 'TAS1',
        'start_date': datetime.fromisoformat('2021-01-01'),
        'end_date': datetime.fromisoformat('2023-04-01'),
    },
    {
        'label': 'vic1_2021_2023',
        'region': 'VIC1',
        'start_date': datetime.fromisoformat('2021-04-01'),
        'end_date': datetime.fromisoformat('2023-12-01'),
    },
]

STEP_DURATION = 5 / 60
EPISODE_HOURS = 24 * 365 * 1.5
ACTION_MODE = 'multi_market'
DEGRADATION_MODE = 'real_world'
DEGRADATION_CHEMISTRY = 'LFP'
DEGRADATION_TEMPERATURE = 30.0
CONTEXT_LENGTH = 1152  # 4 days of 5-minute steps
DATASET_TAG = 'aemo_dt'

CACHE_DIR = REPO_ROOT / 'src' / 'data' / 'aemo'
OUTPUT_DIR = REPO_ROOT / 'data' / 'aemo_dt'
MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'aemo_decision_transformer_model_kwargs.json'
SCENARIO_MANIFEST_PATH = OUTPUT_DIR / f'{DATASET_TAG}_scenario_manifest.json'

BATTERY_VARIANTS = [
    {'name': 'small', 'capacity_mwh': 2.0, 'max_power_mw': 1.0, 'init_soc_ratio': 0.5},
    {'name': 'medium', 'capacity_mwh': 10.0, 'max_power_mw': 5.0, 'init_soc_ratio': 0.5},
    {'name': 'large', 'capacity_mwh': 50.0, 'max_power_mw': 25.0, 'init_soc_ratio': 0.5},
]

BEHAVIOR_RUNS = [
    {
        'policy': 'rule',
        'episodes': 4,
        'battery_variants': ['small', 'medium', 'large'],
        'random_episode_start': True,
        'seed': 42,
    },
    {
        'policy': 'sb3',
        'name': 'a2c',
        'episodes': 2,
        'battery_variants': ['small', 'medium'],
        'algorithm': 'A2C',
        'model_path': REPO_ROOT / 'models' / 'aemo_sb3' /'a2c_aemo_model.zip',
        'deterministic': True,
        'random_episode_start': True,
    },
    {
        'policy': 'sb3',
        'name': 'ddpg',
        'episodes': 2,
        'battery_variants': ['small', 'medium'],
        'algorithm': 'DDPG',
        'model_path': REPO_ROOT / 'models' / 'aemo_sb3' /'ddpg_aemo_model.zip',
        'deterministic': True,
        'random_episode_start': True,
    },
    {
        'policy': 'sb3',
        'name': 'ppo',
        'episodes': 2,
        'battery_variants': ['small', 'medium'],
        'algorithm': 'PPO',
        'model_path': REPO_ROOT / 'models' / 'aemo_sb3' /'ppo_aemo_model.zip',
        'deterministic': True,
        'random_episode_start': True,
    },
    {
        'policy': 'sb3',
        'name': 'sac',
        'episodes': 2,
        'battery_variants': ['small', 'medium'],
        'algorithm': 'SAC',
        'model_path': REPO_ROOT / 'models' / 'aemo_sb3' /'sac_aemo_model.zip',
        'deterministic': True,
        'random_episode_start': True,
    },
    {
        'policy': 'sb3',
        'name': 'td3',
        'episodes': 2,
        'battery_variants': ['small', 'medium'],
        'algorithm': 'TD3',
        'model_path': REPO_ROOT / 'models' / 'aemo_sb3' /'td3_aemo_model.zip',
        'deterministic': True,
        'random_episode_start': True,
    },
]

# Station names below are chosen from dispatch_utils.list_known_batteries().
DISPATCH_RUNS = [
    {
        'label': 'hornsdale_replay',
        'episodes': 1,
        'station_name': 'hornsdale',
        'init_soc_ratio': 0.5,
    },
    {
        'label': 'lake_bonney_replay',
        'episodes': 1,
        'station_name': 'lake_bonney',
        'init_soc_ratio': 0.5,
    },
    {
        'label': 'victorian_big_battery_replay',
        'episodes': 1,
        'station_name': 'victorian_big_battery',
        'init_soc_ratio': 0.5,
    },
    {
        'label': 'wandoan_replay',
        'episodes': 1,
        'station_name': 'wandoan',
        'init_soc_ratio': 0.5,
    },
    {
        'label': 'wallgrove_replay',
        'episodes': 1,
        'station_name': 'wallgrove',
        'init_soc_ratio': 0.5,
    },
]

RUN_DT_TRAINING = False

DT_TRAINING_ARGS = {
    'epochs': 2,
    'batch_size': 8,
    'lr': 2e-5,
    'val_split': 0.1,
    'seed': 8964,
    'device': None,
    'amp_mode': 'off',
    'return_scale': 1.0,
    'action_loss_weight': 1.0,
    'state_loss_weight': 0.01,
    'return_loss_weight': 0.002,
    'weight_decay': 1e-4,
    'num_workers': 2,
    'prefetch_factor': 2,
}

## 2. Fetch and cache multi-scenario AEMO data

In [6]:
run_paths = prepare_run_paths(output_dir=OUTPUT_DIR, dataset_tag=DATASET_TAG)

global_stats, scenario_manifest = fit_aemo_global_stats(
    scenarios=SCENARIOS,
    cache_dir=CACHE_DIR,
    step_duration=STEP_DURATION,
    refresh=False,
)

processed_by_label, _ = fetch_and_preprocess_aemo_scenarios(
    scenarios=SCENARIOS,
    cache_dir=CACHE_DIR,
    step_duration=STEP_DURATION,
    refresh=False,
    fixed_stats=global_stats,
)

scenario_manifest_payload = [
    {
        **entry,
        'start_date': entry['start_date'].isoformat(),
        'end_date': entry['end_date'].isoformat(),
    }
    for entry in scenario_manifest
]

scenario_lookup = {entry['label']: entry for entry in scenario_manifest}

scenario_payloads = [
    (scenario_lookup[entry['label']], processed_by_label[entry['label']])
    for entry in scenario_manifest
]

resolved_battery_variants = resolve_battery_variants(BATTERY_VARIANTS)
resolved_dispatch_runs = resolve_dispatch_replay_runs(DISPATCH_RUNS)
MAX_STEP = int(round(EPISODE_HOURS / STEP_DURATION))
MAX_TIMESTEP = MAX_STEP

write_json(
    SCENARIO_MANIFEST_PATH,
    {
        'global_stats': global_stats,
        'scenarios': scenario_manifest_payload,
    },
)

print(f'scenario count: {len(scenario_manifest)}')
print(f'regions: {sorted({entry["region"] for entry in scenario_manifest})}')
print(f'processed cache dir: {CACHE_DIR}')
pl.DataFrame(scenario_manifest_payload)

Fetching AEMO data bundle for NSW1 from 2021-01-01 to 2023-04-01...
Fetching dispatch price data for NSW1 from 2021-01-01 to 2023-04-01...
INFO: Compiling data for table DISPATCHPRICE
INFO: Returning DISPATCHPRICE.
INFO: Compiling data for table DISPATCHREGIONSUM
INFO: Returning DISPATCHREGIONSUM.
Fetched 239663 price records
Fetching FCAS RAISEREG price data for NSW1 from 2021-01-01 to 2023-04-01...
INFO: Compiling data for table DISPATCHPRICE
INFO: Returning DISPATCHPRICE.
Fetched 239663 FCAS price records
Fetching FCAS LOWERREG price data for NSW1 from 2021-01-01 to 2023-04-01...
INFO: Compiling data for table DISPATCHPRICE
INFO: Returning DISPATCHPRICE.
Fetched 239663 FCAS price records
Fetching FCAS RAISE6SEC price data for NSW1 from 2021-01-01 to 2023-04-01...
INFO: Compiling data for table DISPATCHPRICE
INFO: Returning DISPATCHPRICE.
Fetched 239663 FCAS price records
Fetching FCAS LOWER6SEC price data for NSW1 from 2021-01-01 to 2023-04-01...
INFO: Compiling data for table DISPA

label,region,start_date,end_date,index
str,str,str,str,i64
"""nsw1_2021_2023""","""NSW1""","""2021-01-01T00:00:00""","""2023-04-01T00:00:00""",0
"""qld1_2021_2023""","""QLD1""","""2021-01-01T00:00:00""","""2023-04-01T00:00:00""",1
"""sa1_2022_2023""","""SA1""","""2022-04-01T00:00:00""","""2023-12-01T00:00:00""",2
"""tas1_2021_2023""","""TAS1""","""2021-01-01T00:00:00""","""2023-04-01T00:00:00""",3
"""vic1_2021_2023""","""VIC1""","""2021-04-01T00:00:00""","""2023-12-01T00:00:00""",4


## 3. Collect behavior-policy trajectories per scenario



Rule and SB3 runs are still swept by battery variant. Dispatch replay is configured separately by station so the environment sizing comes from the selected AEMO battery record rather than the generic variant list.

In [7]:
all_logs = {}
raw_outputs = {}

for scenario_entry, scenario_processed_data in scenario_payloads:
    for run in BEHAVIOR_RUNS:
        selected_labels = set(run.get('battery_variants', [variant['label'] for variant in resolved_battery_variants]))
        selected_variants = [variant for variant in resolved_battery_variants if variant['label'] in selected_labels]

        for variant in selected_variants:
            tag = f"{scenario_entry['label']}__{run.get('name', run['policy'])}__{variant['label']}"
            print(f'Collecting {tag}...')

            if run['policy'] == 'rule':
                episodes = run_rule_episodes(
                    processed_data=scenario_processed_data,
                    num_episodes=run['episodes'],
                    battery_capacity=variant['battery_capacity'],
                    max_battery_flow=variant['max_battery_flow'],
                    init_soc=variant['init_soc'],
                    step_duration=STEP_DURATION,
                    battery_life_cost=variant['battery_life_cost'],
                    max_step=MAX_STEP,
                    action_mode=ACTION_MODE,
                    degradation_mode=DEGRADATION_MODE,
                    degradation_chemistry=DEGRADATION_CHEMISTRY,
                    degradation_temperature=DEGRADATION_TEMPERATURE,
                    random_episode_start=run.get('random_episode_start', True),
                    base_seed=run.get('seed', 42),
                )
                dispatch_label = None

            elif run['policy'] == 'sb3':
                episodes = run_sb3_episodes(
                    processed_data=scenario_processed_data,
                    battery_variant=variant,
                    model_path=run['model_path'],
                    algorithm=run['algorithm'],
                    num_episodes=run['episodes'],
                    max_step=MAX_STEP,
                    step_duration=STEP_DURATION,
                    action_mode=ACTION_MODE,
                    degradation_mode=DEGRADATION_MODE,
                    degradation_chemistry=DEGRADATION_CHEMISTRY,
                    degradation_temperature=DEGRADATION_TEMPERATURE,
                    random_episode_start=run.get('random_episode_start', True),
                    deterministic=run.get('deterministic', True),
                )
                dispatch_label = None

            else:
                raise ValueError(f"Unsupported policy in BEHAVIOR_RUNS: {run['policy']}")

            tagged_episodes = [
                episode.with_columns(
                    pl.lit(scenario_entry['label']).alias('scenario_label'),
                    pl.lit(scenario_entry['region']).alias('scenario_region'),
                    pl.lit(scenario_entry['start_date'].isoformat()).alias('scenario_start_date'),
                    pl.lit(scenario_entry['end_date'].isoformat()).alias('scenario_end_date'),
                    pl.lit(run['policy']).alias('policy_name'),
                    pl.lit(variant['label']).alias('battery_label'),
                    pl.lit(dispatch_label, dtype=pl.Utf8).alias('dispatch_label'),
                )
                for episode in episodes
            ]

            raw_path = run_paths['raw_dir'] / f'{tag}_logs.parquet'
            write_combined_episode_logs(episodes=tagged_episodes, output_path=raw_path)
            all_logs[tag] = tagged_episodes
            raw_outputs[tag] = str(raw_path)

    for dispatch_run in resolved_dispatch_runs:
        tag = f"{scenario_entry['label']}__dispatch__{dispatch_run['label']}"
        try:
            should_run_dispatch, dispatch_region = should_run_dispatch_for_scenario(
                scenario_region=scenario_entry['region'],
                dispatch_station=dispatch_run.get('station_name'),
                dispatch_duid=dispatch_run.get('dispatch_duid'),
                start_date=scenario_entry['start_date'],
                end_date=scenario_entry['end_date'],
            )

            if not should_run_dispatch:
                print(
                    f"Skipping {tag}: dispatch target region {dispatch_region} does not match "
                    f"scenario region {scenario_entry['region']}."
                )
                continue

            print(f'Collecting {tag}...')

            selection = build_dispatch_selection(
                region=scenario_entry['region'],
                start_date=scenario_entry['start_date'],
                end_date=scenario_entry['end_date'],
                cache_dir=CACHE_DIR,
                dispatch_station=dispatch_run.get('station_name'),
                dispatch_duid=dispatch_run.get('dispatch_duid'),
                dispatch_index=dispatch_run['dispatch_index'],
                battery_capacity=10.0,
                max_battery_flow=5.0,
                init_soc=0.0,
                init_soc_ratio=dispatch_run['init_soc_ratio'],
            )

            dispatch_battery_life_cost = resolve_dispatch_battery_life_cost(
                dispatch_run=dispatch_run,
                station_capacity_mwh=selection['battery_capacity'],
            )

            episodes, incident_logs, _ = run_dispatch_replay(
                processed_data=scenario_processed_data,
                selection=selection,
                start_date=scenario_entry['start_date'],
                end_date=scenario_entry['end_date'],
                region=scenario_entry['region'],
                cache_dir=str(CACHE_DIR),
                num_episodes=dispatch_run['episodes'],
                step_duration=STEP_DURATION,
                battery_life_cost=dispatch_battery_life_cost,
                max_step=MAX_STEP,
                output_dir=None,
                run_tag=tag,
                action_mode=ACTION_MODE,
                degradation_mode=DEGRADATION_MODE,
                degradation_chemistry=DEGRADATION_CHEMISTRY,
                degradation_temperature=DEGRADATION_TEMPERATURE,
            )
        except ValueError as exc:
            print(f"Skipping {tag}: {exc}")
            continue

        if any(df.height > 0 for df in incident_logs):
            incident_path = run_paths['raw_dir'] / f'{tag}_incident_logs.parquet'
            pl.concat([df.with_columns(pl.lit(i).alias('episode_id')) for i, df in enumerate(incident_logs) if df.height > 0], how='diagonal_relaxed').write_parquet(incident_path)
            raw_outputs[f'{tag}__incidents'] = str(incident_path)

        dispatch_station_label = selection.get('station_key') or selection.get('station_name') or dispatch_run['label']

        tagged_episodes = [
            episode.with_columns(
                pl.lit(scenario_entry['label']).alias('scenario_label'),
                pl.lit(scenario_entry['region']).alias('scenario_region'),
                pl.lit(scenario_entry['start_date'].isoformat()).alias('scenario_start_date'),
                pl.lit(scenario_entry['end_date'].isoformat()).alias('scenario_end_date'),
                pl.lit('dispatch').alias('policy_name'),
                pl.lit(None, dtype=pl.Utf8).alias('battery_label'),
                pl.lit(dispatch_run['label']).alias('dispatch_label'),
                pl.lit(dispatch_station_label).alias('dispatch_station'),
            )
            for episode in episodes
        ]

        raw_path = run_paths['raw_dir'] / f'{tag}_logs.parquet'
        write_combined_episode_logs(episodes=tagged_episodes, output_path=raw_path)
        all_logs[tag] = tagged_episodes
        raw_outputs[tag] = str(raw_path)

print(sorted(all_logs))

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete


/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


...

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete


/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Skipping tas1_2021_2023__dispatch__hornsdale_replay: dispatch target region SA1 does not match scenario region TAS1.
Skipping tas1_2021_2023__dispatch__lake_bonney_replay: dispatch target region SA1 does not match scenario region TAS1.
Skipping tas1_2021_2023__dispatch__victorian_big_battery_replay: dispatch target region VIC1 does not match scenario region TAS1.
Skipping tas1_2021_2023__dispatch__wandoan_replay: dispatch target region QLD1 does not match scenario region TAS1.
Skipping tas1_2021_2023__dispatch__wallgrove_replay: dispatch target region NSW1 does not match scenario region TAS1.
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Sim Complete
Skipping vic1_2021_2023__dispatch__hornsdale_replay: dispatch target region SA1 does not match scenario region VIC1.
Skipping vic1_2021_2023__dispatch__lake_bonney_replay: dispatch target region SA1 does not match scenario region VIC1.
Station:

In [8]:
for scenario_entry, scenario_processed_data in scenario_payloads:
    for dispatch_run in resolved_dispatch_runs:
        tag = f"{scenario_entry['label']}__dispatch__{dispatch_run['label']}"
        try:
            should_run_dispatch, dispatch_region = should_run_dispatch_for_scenario(
                scenario_region=scenario_entry['region'],
                dispatch_station=dispatch_run.get('station_name'),
                dispatch_duid=dispatch_run.get('dispatch_duid'),
                start_date=scenario_entry['start_date'],
                end_date=scenario_entry['end_date'],
            )

            if not should_run_dispatch:
                print(
                    f"Skipping {tag}: dispatch target region {dispatch_region} does not match "
                    f"scenario region {scenario_entry['region']}."
                )
                continue

            print(f'Collecting {tag}...')

            selection = build_dispatch_selection(
                region=scenario_entry['region'],
                start_date=scenario_entry['start_date'],
                end_date=scenario_entry['end_date'],
                cache_dir=CACHE_DIR,
                dispatch_station=dispatch_run.get('station_name'),
                dispatch_duid=dispatch_run.get('dispatch_duid'),
                dispatch_index=dispatch_run['dispatch_index'],
                battery_capacity=10.0,
                max_battery_flow=5.0,
                init_soc=0.0,
                init_soc_ratio=dispatch_run['init_soc_ratio'],
            )

            dispatch_battery_life_cost = resolve_dispatch_battery_life_cost(
                dispatch_run=dispatch_run,
                station_capacity_mwh=selection['battery_capacity'],
            )

            episodes, incident_logs, _ = run_dispatch_replay(
                processed_data=scenario_processed_data,
                selection=selection,
                start_date=scenario_entry['start_date'],
                end_date=scenario_entry['end_date'],
                region=scenario_entry['region'],
                cache_dir=str(CACHE_DIR),
                num_episodes=dispatch_run['episodes'],
                step_duration=STEP_DURATION,
                battery_life_cost=dispatch_battery_life_cost,
                max_step=MAX_STEP,
                output_dir=None,
                run_tag=tag,
                action_mode=ACTION_MODE,
                degradation_mode=DEGRADATION_MODE,
                degradation_chemistry=DEGRADATION_CHEMISTRY,
                degradation_temperature=DEGRADATION_TEMPERATURE,
            )
        except ValueError as exc:
            print(f"Skipping {tag}: {exc}")
            continue

        if any(df.height > 0 for df in incident_logs):
            incident_path = run_paths['raw_dir'] / f'{tag}_incident_logs.parquet'
            pl.concat([df.with_columns(pl.lit(i).alias('episode_id')) for i, df in enumerate(incident_logs) if df.height > 0], how='diagonal_relaxed').write_parquet(incident_path)
            raw_outputs[f'{tag}__incidents'] = str(incident_path)

        dispatch_station_label = selection.get('station_key') or selection.get('station_name') or dispatch_run['label']

        tagged_episodes = [
            episode.with_columns(
                pl.lit(scenario_entry['label']).alias('scenario_label'),
                pl.lit(scenario_entry['region']).alias('scenario_region'),
                pl.lit(scenario_entry['start_date'].isoformat()).alias('scenario_start_date'),
                pl.lit(scenario_entry['end_date'].isoformat()).alias('scenario_end_date'),
                pl.lit('dispatch').alias('policy_name'),
                pl.lit(None, dtype=pl.Utf8).alias('battery_label'),
                pl.lit(dispatch_run['label']).alias('dispatch_label'),
                pl.lit(dispatch_station_label).alias('dispatch_station'),
            )
            for episode in episodes
        ]

        raw_path = run_paths['raw_dir'] / f'{tag}_logs.parquet'
        write_combined_episode_logs(episodes=tagged_episodes, output_path=raw_path)
        all_logs[tag] = tagged_episodes
        raw_outputs[tag] = str(raw_path)

print(sorted(all_logs))

Skipping nsw1_2021_2023__dispatch__hornsdale_replay: dispatch target region SA1 does not match scenario region NSW1.
Skipping nsw1_2021_2023__dispatch__lake_bonney_replay: dispatch target region SA1 does not match scenario region NSW1.
Skipping nsw1_2021_2023__dispatch__victorian_big_battery_replay: dispatch target region VIC1 does not match scenario region NSW1.
Skipping nsw1_2021_2023__dispatch__wandoan_replay: dispatch target region QLD1 does not match scenario region NSW1.
Station: 'Wallgrove BESS 1' (NSW1)
DUIDs active in 2021-01-01 → 2023-04-01: ['WALGRV1']
Fetching unit dispatch data for 1 DUIDs from 2021-01-01 to 2023-04-01...
  DISPATCHLOAD chunk 1/27: 2021-01-01 to 2021-02-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 2/27: 2021-02-01 to 2021-03-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 3/27: 2021-03-01 to 2021-04-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Ret

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 2/20: 2022-05-01 to 2022-06-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 3/20: 2022-06-01 to 2022-07-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 4/20: 2022-07-01 to 2022-08-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 5/20: 2022-08-01 to 2022-09-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 6/20: 2022-09-01 to 2022-10-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 7/20: 2022-10-01 to 2022-11-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 8/20: 2022-11-01 to 2022-12-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 9/20: 2022-12-01 to 2023-01-01
INFO: Compiling data for table DISPA

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 2/20: 2022-05-01 to 2022-06-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 3/20: 2022-06-01 to 2022-07-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 4/20: 2022-07-01 to 2022-08-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 5/20: 2022-08-01 to 2022-09-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 6/20: 2022-09-01 to 2022-10-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 7/20: 2022-10-01 to 2022-11-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 8/20: 2022-11-01 to 2022-12-01
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
  DISPATCHLOAD chunk 9/20: 2022-12-01 to 2023-01-01
INFO: Compiling data for table DISPA

## 4. Build the DT dataset and manifest

The dataset manifest now records the scenario list, shared normalization stats, and the per-run raw log outputs.

In [10]:
dataset, manifest = build_dt_dataset_from_logs(all_logs)
validate_aemo_dt_dimensions(manifest, action_mode=ACTION_MODE)
model_kwargs = build_model_config(
    action_mode=ACTION_MODE,
    context_len=CONTEXT_LENGTH,
    max_timestep=MAX_TIMESTEP,
    output_path=MODEL_CONFIG_PATH,
)

dataset.write_parquet(run_paths['dataset_path'])

behavior_runs_json = [
    {
        **run,
        'model_path': str(run['model_path']) if run.get('model_path') is not None else None,
    }
    for run in BEHAVIOR_RUNS
]

dispatch_runs_json = [
    {
        **run,
    }
    for run in resolved_dispatch_runs
]

manifest.update({
    'created_at': datetime.now(timezone.utc).isoformat(),
    'dataset_path': str(run_paths['dataset_path']),
    'manifest_path': str(run_paths['manifest_path']),
    'scenario_manifest_path': str(SCENARIO_MANIFEST_PATH),
    'cache_dir': str(CACHE_DIR),
    'region_count': len({entry['region'] for entry in scenario_manifest}),
    'scenario_count': len(scenario_manifest),
    'global_stats': global_stats,
    'scenarios': scenario_manifest_payload,
    'step_duration': STEP_DURATION,
    'episode_hours': EPISODE_HOURS,
    'max_step': MAX_STEP,
    'action_mode': ACTION_MODE,
    'degradation_mode': DEGRADATION_MODE,
    'degradation_chemistry': DEGRADATION_CHEMISTRY,
    'degradation_temperature': DEGRADATION_TEMPERATURE,
    'battery_variants': resolved_battery_variants,
    'behavior_runs': behavior_runs_json,
    'dispatch_runs': dispatch_runs_json,
    'model_config_path': str(MODEL_CONFIG_PATH),
    'model_kwargs': model_kwargs,
    'raw_outputs': raw_outputs,
})

write_json(run_paths['manifest_path'], manifest)

print(run_paths['dataset_path'])
print(run_paths['manifest_path'])
dataset.head()

/code/data/aemo_dt/aemo_dt_dataset.parquet
/code/data/aemo_dt/aemo_dt_manifest.json


step,norm_observation,action,reward,episode_id,source_policy
i64,list[f32],list[f32],f32,i32,str
0,"[0.896873, -0.442289, … 0.5]","[-0.001093, 0.0, 0.0]",-0.000029,0,"""nsw1_2021_2023__rule__small"""
1,"[0.887011, -0.461749, … 0.499954]","[-0.002303, 0.0, 0.0]",-0.000026,0,"""nsw1_2021_2023__rule__small"""
2,"[0.876727, -0.480989, … 0.499859]","[-0.001159, 0.0, 0.0]",-0.000029,0,"""nsw1_2021_2023__rule__small"""
3,"[0.866025, -0.5, … 0.49981]","[0.007836, 0.0, 0.0]",-0.000056,0,"""nsw1_2021_2023__rule__small"""
4,"[0.854912, -0.518773, … 0.500137]","[-0.002958, 0.0, 0.0]",-0.000024,0,"""nsw1_2021_2023__rule__small"""


## 5. Optional: launch Decision Transformer training

Set `RUN_DT_TRAINING = True` in the config cell if you want the notebook to call the trainer.

In [ ]:
if RUN_DT_TRAINING:
    command = [
        'python3',
        'src/pretrain_aemo_decision_transformer.py',
        '--dataset-path', str(run_paths['dataset_path']),
        '--model-config', str(MODEL_CONFIG_PATH),
        '--save-path', str(run_paths['output_dir'] / 'aemo_dt_model.pt'),
        '--checkpoint-path', str(run_paths['output_dir'] / 'aemo_dt_checkpoint.pt'),
        '--loss-csv-path', str(run_paths['output_dir'] / 'aemo_dt_loss_history.csv'),
        '--epochs', str(DT_TRAINING_ARGS['epochs']),
        '--batch-size', str(DT_TRAINING_ARGS['batch_size']),
        '--lr', str(DT_TRAINING_ARGS['lr']),
        '--val-split', str(DT_TRAINING_ARGS['val_split']),
        '--seed', str(DT_TRAINING_ARGS['seed']),
        '--amp-mode', DT_TRAINING_ARGS['amp_mode'],
        '--return-scale', str(DT_TRAINING_ARGS['return_scale']),
        '--action-loss-weight', str(DT_TRAINING_ARGS['action_loss_weight']),
        '--state-loss-weight', str(DT_TRAINING_ARGS['state_loss_weight']),
        '--return-loss-weight', str(DT_TRAINING_ARGS['return_loss_weight']),
        '--weight-decay', str(DT_TRAINING_ARGS['weight_decay']),
        '--num-workers', str(DT_TRAINING_ARGS['num_workers']),
        '--prefetch-factor', str(DT_TRAINING_ARGS['prefetch_factor']),
    ]
    print('AEMO DT training command:')
    print(' '.join(command))
else:
    print('DT training skipped. Set RUN_DT_TRAINING = True to print the AEMO training command.')